In [1]:
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
from scipy.signal import find_peaks
from itertools import product
from tqdm import tqdm
import time
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import talib as ta
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
pd.set_option('display.float_format', '{:.7f}'.format)

In [3]:
df=pd.read_csv('../../all_data_EUR_USD.csv')

In [4]:
df.set_index('time',inplace=True)

In [5]:
#df['MACD_low'] = ta.MACD(df['l'], fastperiod=12, slowperiod=26, signalperiod=9)[2]

In [6]:
#df['MACD_high'] = ta.MACD(df['h'], fastperiod=12, slowperiod=26, signalperiod=9)[2]

In [7]:
df_test=df.copy()

In [8]:
def calc_low_points(low_vals):

    temp_min=low_vals[0]
    

    for i in range(len(low_vals)):

         if low_vals[i]<temp_min:

             temp_min=low_vals[i]


        

    if temp_min==low_vals[-2]:

       return low_vals[-2]

    else:

        return -101

In [9]:
def calc_high_points(high_vals):

    temp_max=high_vals[0]
    

    for i in range(len(high_vals)):

         if high_vals[i]>temp_max:

             temp_max=high_vals[i]


        

    if temp_max==high_vals[-2]:

       return high_vals[-2]

    else:

        return 101

In [10]:
def find_ll(macd):


    temp_min=-101
   

    for i in range(len(macd) - 2, -1, -1):
        if macd[i]!=-101:
       
            if macd[i]<macd[-1]:
    
                temp_min=macd[i]
                
                break

               
    
    return temp_min

In [11]:
def find_hh(macd):

    temp_max=101

    for i in range(len(macd) - 2, -1, -1):
        if macd[i]!=101:
       
            if macd[i]>macd[-1]:
    
                temp_max=macd[i]
                
                break

               
    
    return temp_max

In [12]:
def find_ll_idx(macd):

    
    last_min_idx=0

    for i in range(len(macd) - 2, -1, -1):
        if macd[i]!=-101:
        
            if macd[i]<macd[-1]:
    
                
                last_min_idx=i-1
                break

               
    
    return last_min_idx

In [13]:
def find_hh_idx(macd):

    
    last_max_idx=0

    for i in range(len(macd) - 2, -1, -1):
        if macd[i]!=101:
        
            if macd[i]>macd[-1]:
        
                
                last_max_idx=i-1
                break

               
    
    return last_max_idx

In [14]:
class MACD_divergence():

    def __init__(self, data):

        self.data=data

        self.params_range={'macd_fast':[8,12,20],
                           'macd_slow':[20,26,38],
                           'macd_signal':[6,9,15],
                          'lookback':[36,57,91]}
                          


        self.possible_strats={'strategy_desc':'Strategy MACD divergence',
                              'div_point':{'name':'detect divergence MACD points',
                                          'positions':{'buy':'bull point',
                                                       'sell':'bear point'},
                                            'pos_columns':{}}}

    def create_params_combs(self):

        return list(product(*self.params_range.values()))

    def calc_indicator(self, macd_fast,macd_slow,macd_signal, lookback):


        self.macd_fast=macd_fast
        self.macd_slow=macd_slow
        self.macd_signal=macd_signal
        self.lookback=lookback
        
        df=self.data.copy()
        
        
            
        df['MACD_high']=ta.MACD(df['h'],macd_fast,macd_slow,macd_signal)[2]
        df['MACD_low']=ta.MACD(df['l'],macd_fast,macd_slow,macd_signal)[2]
        
        
        
        
        
             
        
        
        
        df['ll']=df['MACD_low'].rolling(3).apply(calc_low_points, raw=True, engine='numba')
        df['hh']=df['MACD_high'].rolling(3).apply(calc_high_points, raw=True, engine='numba')
        
        
        
        df['idx']=range(len(df))
        
        df['last_min']=df['ll'].rolling(lookback).apply(find_ll, raw=True, engine='numba')
        df['last_max']=df['hh'].rolling(lookback).apply(find_hh, raw=True, engine='numba')
        
        df['prev_min_idx']=df['ll'].rolling(lookback).apply(find_ll_idx, raw=True, engine='numba')+df.idx-lookback+1
        df['prev_max_idx']=df['hh'].rolling(lookback).apply(find_hh_idx, raw=True, engine='numba')+df.idx-lookback+1
        df['last_min_idx']=np.where((df['ll']!=-101)&(df['last_min']!=-101)&(df['ll'].notna()), df['idx'].shift(),0)
        df['last_max_idx']=np.where((df['hh']!=101)&(df['last_max']!=101)&(df['hh'].notna()), df['idx'].shift(),0)
        df['last_min_price']=np.where((df['ll']!=-101)&(df['last_min']!=-101)&(df['ll'].notna()), df['l'].shift(),0)
        df['last_max_price']=np.where((df['hh']!=101)&(df['last_max']!=101)&(df['hh'].notna()), df['h'].shift(),0)
        
        prev_min_prices_idx=np.array(df['prev_min_idx'])
        prev_max_prices_idx=np.array(df['prev_max_idx'])
        prev_min_prices=np.array(df['l'].iloc[np.nan_to_num(prev_min_prices_idx)])
        prev_max_prices=np.array(df['h'].iloc[np.nan_to_num(prev_max_prices_idx)])
        df['prev_min_prices']=prev_min_prices
        df['prev_max_prices']=prev_max_prices
        
        self.data=df.copy()

    def calc_position(self):

        df=self.data.copy()

        self.pos_ch_colname=f'pos_ch_div_point_mcdf_{self.macd_fast}_mcdsl_{self.macd_slow}_mcdsi_{self.macd_signal}_lb_{self.lookback}'
        self.pos_colname=f'pos_div_point_mcdf_{self.macd_fast}_mcdsl_{self.macd_slow}_mcdsi_{self.macd_signal}_lb_{self.lookback}'

        cond_buy_check=(df['ll']!=-101)&(df['last_min']!=-101)&(df['prev_min_idx'].notna())
        cond_buy_price=(df['last_min_price']!=0)&(df['prev_min_prices']>df['last_min_price'])
        cond_sell_check=(df['hh']!=101)&(df['last_max']!=101)&(df['prev_max_idx'].notna())
        cond_sell_price=(df['last_max_price']!=0)&(df['prev_max_prices']<df['last_max_price'])

        df[f'{self.pos_ch_colname}']=np.nan
        df[f'{self.pos_ch_colname}']=np.where(cond_buy_check&cond_buy_price,1,df[f'{self.pos_ch_colname}'])
        df[f'{self.pos_ch_colname}']=np.where(cond_sell_check&cond_sell_price,-1,df[f'{self.pos_ch_colname}'])
        df[f'{self.pos_colname}']= df[f'{self.pos_ch_colname}'].ffill()
        df[f'{self.pos_ch_colname}']=df[f'{self.pos_ch_colname}'].fillna(0)
        df[f'{self.pos_colname}']= df[f'{self.pos_colname}'].fillna(0)
        
        

        self.data=df[[col for col in df.columns if col not in ['ll','hh','idx','last_min',
                                                              'last_max','prev_min_idx','prev_max_idx','last_min_idx','last_max_idx','last_min_price','last_max_price',
                                                              'prev_min_prices','prev_max_prices']]]

In [15]:
mcd=MACD_divergence(df_test)

In [16]:
cmb=mcd.create_params_combs()

In [17]:
cmb

[(8, 20, 6, 36),
 (8, 20, 6, 57),
 (8, 20, 6, 91),
 (8, 20, 9, 36),
 (8, 20, 9, 57),
 (8, 20, 9, 91),
 (8, 20, 15, 36),
 (8, 20, 15, 57),
 (8, 20, 15, 91),
 (8, 26, 6, 36),
 (8, 26, 6, 57),
 (8, 26, 6, 91),
 (8, 26, 9, 36),
 (8, 26, 9, 57),
 (8, 26, 9, 91),
 (8, 26, 15, 36),
 (8, 26, 15, 57),
 (8, 26, 15, 91),
 (8, 38, 6, 36),
 (8, 38, 6, 57),
 (8, 38, 6, 91),
 (8, 38, 9, 36),
 (8, 38, 9, 57),
 (8, 38, 9, 91),
 (8, 38, 15, 36),
 (8, 38, 15, 57),
 (8, 38, 15, 91),
 (12, 20, 6, 36),
 (12, 20, 6, 57),
 (12, 20, 6, 91),
 (12, 20, 9, 36),
 (12, 20, 9, 57),
 (12, 20, 9, 91),
 (12, 20, 15, 36),
 (12, 20, 15, 57),
 (12, 20, 15, 91),
 (12, 26, 6, 36),
 (12, 26, 6, 57),
 (12, 26, 6, 91),
 (12, 26, 9, 36),
 (12, 26, 9, 57),
 (12, 26, 9, 91),
 (12, 26, 15, 36),
 (12, 26, 15, 57),
 (12, 26, 15, 91),
 (12, 38, 6, 36),
 (12, 38, 6, 57),
 (12, 38, 6, 91),
 (12, 38, 9, 36),
 (12, 38, 9, 57),
 (12, 38, 9, 91),
 (12, 38, 15, 36),
 (12, 38, 15, 57),
 (12, 38, 15, 91),
 (20, 20, 6, 36),
 (20, 20, 6, 57),
 

In [18]:
#mcd.calc_indicator(12,26,9,30)

In [19]:
#mcd.data

In [20]:
#mcd.calc_position()

In [21]:
#mcd.data

In [22]:
for c in tqdm(cmb):

    

    mcd.calc_indicator(*c)
    
    mcd.calc_position()

100%|██████████| 81/81 [04:00<00:00,  2.97s/it]


In [23]:
mcd.data.to_csv('macd_divergence_data.csv')

In [24]:
with open('macd_divergence.json', "w") as f:
    json.dump(mcd.possible_strats, f)

In [25]:
df_plot=mcd.data.copy()

In [26]:
figure = make_subplots(rows=2, cols=1, row_heights=[0.7,0.3], shared_xaxes=True,vertical_spacing=0.01)
figure.update_layout(height=800, width=1200, title_text='RSI_divergence')

pos_col='pos_ch_div_point_mcdf_12_mcdsl_26_mcdsi_9_lb_30'

y_color_buy=df_plot[df_plot[pos_col]==1]['l']*0.9998
y_color_buy_index=df_plot[df_plot[pos_col]==1].index
y_color_sell=df_plot[df_plot[pos_col]==-1]['h']*1.0002
y_color_sell_index=df_plot[df_plot[pos_col]==-1].index

#y_color_sell=df_plot[df_plot['pos_ch_bear']==-1]['h']*1.0002
#y_color_sell_index=df_plot[df_plot['pos_ch_bear']==-1].index
#y_color_buy=df_plot[df_plot['pos_ch_bull']==1]['l']*0.9998
#y_color_buy_index=df_plot[df_plot['pos_ch_bull']==1].index


figure.add_trace(go.Candlestick(x=df_plot.index,
                                open=df_plot['o'],
                                high=df_plot['h'],
                                low=df_plot['l'],
                                close=df_plot['c'],
                                name='price'), row=1, col=1)

figure.add_trace(go.Scatter(x=df_plot.index,y=df_plot['MACD_low'],mode='lines',line_color='green',name='macdhist'),col=1,row=2)
figure.add_trace(go.Scatter(x=df_plot.index,y=df_plot['MACD_high'],mode='lines',line_color='red',name='macdhist'),col=1,row=2)
figure.add_trace(go.Scatter(x=y_color_buy_index, y=y_color_buy, mode='markers', marker_symbol='arrow-up', marker_color='green', name='buy', marker_size=10), col=1, row=1)
figure.add_trace(go.Scatter(x=y_color_sell_index, y=y_color_sell, mode='markers', marker_symbol='arrow-down', marker_color='red', name='sell', marker_size=10), col=1, row=1)
#for i in idxs:
    

    #figure.add_trace(go.Scatter(x=df_plot.loc[i].index,y=df_plot['RSI_low'].loc[i],mode='lines',line_color='red',name='macdhist'),col=1,row=2)

    

figure.update_layout(xaxis_rangeslider_visible=False)
figure.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "mon"])]
    )
figure.show()

KeyError: 'pos_ch_div_point_mcdf_12_mcdsl_26_mcdsi_9_lb_30'